# 🔎 GraphShields — Notebook 2: Explainability Generation (No Retraining)
AMAD Hackathon · loads `results_final/` artifacts only · outputs to `results_final/explainability/` and `results_final/website/`

**This notebook never retrains, reshuffles, re-thresholds, or re-evaluates.**
All models, predictions, thresholds, and splits are final from Notebook 1.

Sections: 1 Setup · 2 Load Artifacts · 3 SHAP · 4 GNNExplainer · 5 Visualization Data · 6 Website Export


## 1. Setup

In [1]:
import os, json, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure torch-geometric and its dependencies are properly installed for the current Colab environment.
# This multi-step installation is often necessary for PyG, specifically to fetch
# CUDA-compatible wheels for torch-scatter, torch-sparse, etc.
TORCH_VERSION = torch.__version__.split('+')[0] # Get base version, e.g., "2.0.0" from "2.0.0+cu118"

if torch.cuda.is_available():
    CUDA_VERSION = 'cu' + torch.version.cuda.replace('.', '') # e.g., "cu118" from "11.8"
    # Construct the URL for PyG's custom wheel index
    PYG_WHL_URL = f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html"
    print(f"Detected PyTorch version {torch.__version__} with CUDA. Installing PyG dependencies from: {PYG_WHL_URL}")
    !pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f {PYG_WHL_URL}
else:
    # Fallback for CPU-only environments. This path is less common in Colab for GPU-accelerated PyG.
    print(f"Detected PyTorch version {torch.__version__} on CPU. Installing PyG dependencies for CPU.")
    !pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric

# Install shap, ensuring it's available. This was part of the original pip install attempt.
!pip install -q shap

# Now import the modules that previously failed
from torch.nn import BatchNorm1d, LeakyReLU, Dropout
from torch_geometric.nn import GATv2Conv
from torch_geometric.explain import Explainer, GNNExplainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Detected PyTorch version 2.11.0+cu128 with CUDA. Installing PyG dependencies from: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 143.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 39.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 40.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.4 MB/s eta 0:00:00
Device: cuda


## 2. Load All Artifacts from Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE   = "/content/drive/MyDrive"
RESULTS_ROOT = f"{DRIVE_BASE}/GraphShields/results_final"
SHARED_ROOT  = f"{DRIVE_BASE}/GraphShields/shared"

PATHS = {
    "xgb_model":          f"{RESULTS_ROOT}/models/xgb_model.pkl",
    "gatv2_weights":      f"{RESULTS_ROOT}/models/gatv2_weights.pt",
    "pyg_graph":          f"{RESULTS_ROOT}/models/pyg_graph.pt",
    "tx_mapping":         f"{RESULTS_ROOT}/graph/transaction_mapping.csv",
    "hybrid_preds":       f"{RESULTS_ROOT}/predictions/hybrid_predictions.csv",
    "gatv2_preds":        f"{RESULTS_ROOT}/predictions/gatv2_predictions.csv",
    "xgb_preds":          f"{RESULTS_ROOT}/predictions/xgb_predictions.csv",
    "xgb_test_features":  f"{RESULTS_ROOT}/predictions/xgb_test_features.csv",
    "best_threshold":     f"{RESULTS_ROOT}/threshold/best_threshold.json",
    "gat_embeddings":     f"{RESULTS_ROOT}/embeddings/gat_embeddings.npy",
    "feature_categories": f"{SHARED_ROOT}/feature_categories.json",
}

missing = [k for k, p in PATHS.items() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Missing artifacts — run Notebook 1 first: {missing}")
print("All artifacts found.")


Mounted at /content/drive
All artifacts found.


In [3]:
# --- Load everything ---
xgb_model      = joblib.load(PATHS["xgb_model"])
gat_embeddings = np.load(PATHS["gat_embeddings"])
graph          = torch.load(PATHS["pyg_graph"], map_location="cpu", weights_only=False)
tx_mapping_df  = pd.read_csv(PATHS["tx_mapping"])          # node_index, txId
hybrid_df      = pd.read_csv(PATHS["hybrid_preds"])        # txId, true_label, prob, pred
gatv2_df       = pd.read_csv(PATHS["gatv2_preds"])
xgb_df         = pd.read_csv(PATHS["xgb_preds"])
xgb_features   = pd.read_csv(PATHS["xgb_test_features"])  # txId + V1…V166

with open(PATHS["best_threshold"]) as f:
    threshold_data = json.load(f)
THRESHOLD = threshold_data["best_threshold"]

with open(PATHS["feature_categories"]) as f:
    feat_cats = json.load(f)

# txId <-> node_index lookup
node_to_tx = dict(zip(tx_mapping_df["node_index"], tx_mapping_df["txId"]))
tx_to_node = dict(zip(tx_mapping_df["txId"], tx_mapping_df["node_index"]))

FEAT_COLS = [c for c in xgb_features.columns if c != "txId"]

print(f"Loaded. Test transactions: {len(hybrid_df):,}  |  Threshold: {THRESHOLD}")
print(f"Graph: {graph.num_nodes:,} nodes / {graph.num_edges:,} edges")
print(f"Feature categories: {[r['label'] for r in feat_cats['ranges']]}")


Loaded. Test transactions: 11,184  |  Threshold: 0.8
Graph: 203,769 nodes / 234,355 edges
Feature categories: ['Transaction Profile', 'Network Context', 'Hidden Topological Patterns']


In [4]:
# Weber et al. 2019 feature category mapping (from Notebook 4 / shared/feature_categories.json)
def get_category(feat_name: str) -> dict:
    """Return category info for a feature name like V43 or emb_5."""
    if feat_name.startswith("emb_"):
        idx, prefix = int(feat_name.split("_", 1)[1]), "emb"
    elif feat_name.startswith("V"):
        idx, prefix = int(feat_name[1:]), "V"
    else:
        return {"label": "Other", "description": "Unknown feature type"}
    for r in feat_cats["ranges"]:
        if r["prefix"] == prefix and r["start"] <= idx <= r["end"]:
            return r
    return {"label": "Other", "description": "Outside defined range"}

def humanize_shap_entries(feat_names, shap_vals, top_k=5, direction="positive"):
    """Return top-k SHAP entries as structured dicts with category info."""
    sign = 1 if direction == "positive" else -1
    order = np.argsort(sign * shap_vals)[::-1][:top_k]
    results = []
    for i in order:
        cat = get_category(feat_names[i])
        results.append({
            "feature":   feat_names[i],
            "category":  cat["label"],
            "impact":    f"{shap_vals[i]:+.4f}",
            "direction": "increased_risk" if shap_vals[i] > 0 else "decreased_risk",
        })
    return results


## 3. SHAP Explainability (XGBoost)
Top 20 most suspicious (by hybrid risk score) + top 5 most confident normal = **25 transactions total**.
Feature labels use the Weber et al. category mapping — no raw "V43" strings in the final output.

In [5]:
# Select 25 transactions: top 20 suspicious + top 5 confident-normal
top_suspicious = hybrid_df.sort_values("prob", ascending=False).head(20)
top_normal     = hybrid_df.sort_values("prob", ascending=True).head(5)
selected       = pd.concat([top_suspicious, top_normal]).drop_duplicates("txId")

selected_txids  = set(selected["txId"])
xgb_sel         = xgb_features[xgb_features["txId"].isin(selected_txids)].copy()
xgb_sel_ordered = selected.merge(xgb_sel, on="txId", how="left")

X_sel = xgb_sel_ordered[FEAT_COLS].values
print(f"Explaining {len(X_sel)} transactions ({len(top_suspicious)} suspicious + {len(top_normal)} normal)")


Explaining 25 transactions (20 suspicious + 5 normal)


In [6]:
# SHAP TreeExplainer — runs only on the 25 selected rows, not the full test set
explainer_shap = shap.TreeExplainer(xgb_model)
shap_values    = explainer_shap.shap_values(X_sel)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # binary: take positive-class SHAP

print(f"SHAP values shape: {shap_values.shape}")


SHAP values shape: (25, 166)


In [7]:
# Build output dirs
EXPL_ROOT  = f"{RESULTS_ROOT}/explainability"
SHAP_DIR   = f"{EXPL_ROOT}/shap"
GNN_DIR    = f"{EXPL_ROOT}/gnn"
VIZ_DIR    = f"{RESULTS_ROOT}/visualization"
WEB_DIR    = f"{RESULTS_ROOT}/website"
WEB_EXPL   = f"{WEB_DIR}/explanations"
for d in [SHAP_DIR, GNN_DIR, VIZ_DIR, WEB_DIR, WEB_EXPL]:
    os.makedirs(d, exist_ok=True)

# shap_summary.png
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sel, feature_names=FEAT_COLS, show=False)
plt.tight_layout()
plt.savefig(f"{SHAP_DIR}/shap_summary.png", dpi=150, bbox_inches="tight")
plt.close()

# shap_bar.png
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sel, feature_names=FEAT_COLS, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(f"{SHAP_DIR}/shap_bar.png", dpi=150, bbox_inches="tight")
plt.close()

print("Saved shap_summary.png and shap_bar.png")


Saved shap_summary.png and shap_bar.png


In [8]:
# transaction_explanations.csv — structured, readable, category-labeled
records = []
hybrid_lookup = hybrid_df.set_index("txId").to_dict(orient="index")

for row_pos, row in enumerate(xgb_sel_ordered.itertuples()):
    sv = shap_values[row_pos]
    pos_factors = humanize_shap_entries(FEAT_COLS, sv, top_k=5, direction="positive")
    neg_factors = humanize_shap_entries(FEAT_COLS, sv, top_k=5, direction="negative")
    h = hybrid_lookup.get(row.txId, {})
    records.append({
        "txId":              row.txId,
        "hybrid_score":      round(float(h.get("prob", 0)), 4),
        "prediction":        "suspicious" if h.get("pred", 0) == 1 else "normal",
        "top_positive_factors": json.dumps(pos_factors),
        "top_negative_factors": json.dumps(neg_factors),
    })

expl_df = pd.DataFrame(records)
expl_df.to_csv(f"{SHAP_DIR}/transaction_explanations.csv", index=False)
print(f"Saved transaction_explanations.csv ({len(expl_df)} rows)")
print(expl_df[["txId","hybrid_score","prediction"]].head(3).to_string(index=False))


Saved transaction_explanations.csv (25 rows)
    txId  hybrid_score prediction
71989979        0.9973 suspicious
72631257        0.9929 suspicious
72601861        0.9923 suspicious


## 4. GNNExplainer (GATv2)
Top 20 suspicious transactions · 2-hop neighborhood · max 20 neighbors per target.

In [9]:
# Rebuild GATv2 architecture (weights only — model never retrained here)
class GraphShieldsGATv2(torch.nn.Module):
    def __init__(self, in_channels, hidden=16, heads=8, dropout=0.3):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden, heads=heads, dropout=dropout)
        self.bn1   = BatchNorm1d(hidden * heads)
        self.act   = LeakyReLU(0.2)
        self.drop  = Dropout(dropout)
        self.conv2 = GATv2Conv(hidden * heads, 1, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x); x = self.act(x); x = self.drop(x)
        return self.conv2(x, edge_index)

gat = GraphShieldsGATv2(in_channels=graph.num_features).to(device)
gat.load_state_dict(torch.load(PATHS["gatv2_weights"], map_location=device))
gat.to(device).eval()
graph = graph.to(device)
print("GATv2 loaded in eval mode — weights frozen, no retraining.")


GATv2 loaded in eval mode — weights frozen, no retraining.


In [10]:
TOP_N_GNN    = 20
MAX_NEIGHBORS = 20

top20_txids = hybrid_df.sort_values("prob", ascending=False).head(TOP_N_GNN)["txId"].tolist()
top20_nodes = [tx_to_node[t] for t in top20_txids if t in tx_to_node]

edge_np  = graph.edge_index.cpu().numpy()
src_arr, dst_arr = edge_np[0], edge_np[1]

gnn_explainer = Explainer(
    model=gat,
    algorithm=GNNExplainer(epochs=200),
    explanation_type="model",
    node_mask_type="object",
    edge_mask_type="object",
    model_config=dict(mode="binary_classification", task_level="node", return_type="raw"),
)

all_edge_rows, all_node_rows, target_info = [], [], []

for t_node in top20_nodes:
    expl = gnn_explainer(graph.x, graph.edge_index, index=t_node)
    e_mask = expl.edge_mask.detach().cpu().numpy()
    n_mask = expl.node_mask.detach().cpu().numpy().flatten()

    # 2-hop neighborhood: direct neighbors + their neighbors
    hop1  = set(dst_arr[src_arr == t_node]) | set(src_arr[dst_arr == t_node])
    hop2  = set()
    for n in hop1:
        hop2 |= set(dst_arr[src_arr == n]) | set(src_arr[dst_arr == n])
    neighborhood = (hop1 | hop2) - {t_node}

    # Incident edges within 2-hop, sorted by importance, capped
    incident = [i for i in range(len(src_arr))
                if src_arr[i] in neighborhood | {t_node}
                and dst_arr[i] in neighborhood | {t_node}]
    incident = sorted(incident, key=lambda i: -e_mask[i])[:MAX_NEIGHBORS]

    target_info.append({"node_idx": int(t_node), "txId": int(node_to_tx.get(t_node, -1))})

    seen = set()
    for i in incident:
        s, d = int(src_arr[i]), int(dst_arr[i])
        all_edge_rows.append({
            "explained_target_node_idx": int(t_node),
            "explained_target_txId":     int(node_to_tx.get(t_node, -1)),
            "source_node_idx": s, "target_node_idx": d,
            "source_txId": int(node_to_tx.get(s, -1)),
            "target_txId": int(node_to_tx.get(d, -1)),
            "edge_importance": float(e_mask[i]),
        })
        for n in (s, d):
            if n != t_node and n not in seen:
                seen.add(n)
                all_node_rows.append({
                    "explained_target_node_idx": int(t_node),
                    "explained_target_txId":     int(node_to_tx.get(t_node, -1)),
                    "node_idx":       n,
                    "txId":           int(node_to_tx.get(n, -1)),
                    "node_importance": float(n_mask[n] if n < len(n_mask) else 0),
                })

print(f"GNNExplainer done: {len(target_info)} targets, "
      f"{len(all_node_rows)} neighbor rows, {len(all_edge_rows)} edge rows")


GNNExplainer done: 20 targets, 194 neighbor rows, 195 edge rows


In [11]:
important_nodes_df = pd.DataFrame(all_node_rows).sort_values(
    ["explained_target_node_idx","node_importance"], ascending=[True,False]).reset_index(drop=True)
important_edges_df = pd.DataFrame(all_edge_rows).sort_values(
    ["explained_target_node_idx","edge_importance"], ascending=[True,False]).reset_index(drop=True)

important_nodes_df.to_csv(f"{GNN_DIR}/important_nodes.csv", index=False)
important_edges_df.to_csv(f"{GNN_DIR}/important_edges.csv", index=False)

# explanation_graph.json — per-target subgraph for the investigation click-through view
explanation_graph = {
    "purpose": "per-transaction investigation view — loaded when a node is clicked in the dashboard",
    "targets": target_info,
    "nodes": important_nodes_df.rename(columns={
        "explained_target_node_idx":"for_target_node_idx",
        "explained_target_txId":"for_target_txId"
    }).to_dict(orient="records"),
    "edges": important_edges_df.rename(columns={
        "explained_target_node_idx":"for_target_node_idx",
        "explained_target_txId":"for_target_txId"
    }).to_dict(orient="records"),
}
with open(f"{GNN_DIR}/explanation_graph.json","w") as f:
    json.dump(explanation_graph, f, indent=2)
print("Saved important_nodes.csv, important_edges.csv, explanation_graph.json")


Saved important_nodes.csv, important_edges.csv, explanation_graph.json


## 5. Visualization Data — fraud_network.json + risk_nodes.csv + explanation_summary.json

In [12]:
# fraud_network.json — global dashboard graph (all test transactions)
# Contains risk scores, predictions, and metadata. No .pt dependency.
hybrid_lookup2 = hybrid_df.set_index("txId")

viz_nodes, viz_edges = [], []
for _, row in tx_mapping_df.iterrows():
    txid = int(row["txId"])
    n    = int(row["node_index"])
    h    = hybrid_lookup2.loc[txid] if txid in hybrid_lookup2.index else None
    viz_nodes.append({
        "id":         txid,
        "node_index": n,
        "risk":       round(float(h["prob"]), 4) if h is not None else None,
        "prediction": ("suspicious" if h["pred"]==1 else "normal") if h is not None else "unlabeled",
        "true_label": int(h["true_label"]) if h is not None else None,
    })

# Only include edges where both endpoints have predictions (test nodes)
test_txids = set(hybrid_df["txId"])
ei = graph.edge_index.cpu().numpy()
included = set()
for i in range(ei.shape[1]):
    s_tx = node_to_tx.get(int(ei[0,i]))
    d_tx = node_to_tx.get(int(ei[1,i]))
    if s_tx in test_txids and d_tx in test_txids:
        key = (int(s_tx), int(d_tx))
        if key not in included:
            viz_edges.append({"source": int(s_tx), "target": int(d_tx)})
            included.add(key)

fraud_network = {"nodes": viz_nodes, "edges": viz_edges}
with open(f"{VIZ_DIR}/fraud_network.json", "w") as f:
    json.dump(fraud_network, f)
print(f"fraud_network.json: {len(viz_nodes)} nodes, {len(viz_edges)} edges")

# risk_nodes.csv — flat table for quick dashboard filtering
hybrid_df.rename(columns={"prob":"risk_score","pred":"predicted_label"}).to_csv(
    f"{VIZ_DIR}/risk_nodes.csv", index=False)

# explanation_summary.json — per-transaction rich summary for investigation panel
shap_lookup = expl_df.set_index("txId").to_dict(orient="index")
gnn_lookup  = {t["txId"]: [] for t in target_info}
for _, row in important_nodes_df.iterrows():
    gnn_lookup[row["explained_target_txId"]].append({
        "txId": int(row["txId"]), "importance": round(float(row["node_importance"]),4)
    })

summary_records = []
for _, row in hybrid_df.sort_values("prob", ascending=False).iterrows():
    txid  = int(row["txId"])
    shap_r = shap_lookup.get(txid, {})
    neighbors = gnn_lookup.get(txid, [])

    pos_factors = json.loads(shap_r.get("top_positive_factors", "[]"))
    neg_factors = json.loads(shap_r.get("top_negative_factors", "[]"))

    # Human-readable reasons list for the investigation panel
    reasons = []
    for f in pos_factors[:3]:
        reasons.append(f"{f['category']} increased risk ({f['impact']}, feature {f['feature']})")
    for f in neg_factors[:2]:
        reasons.append(f"{f['category']} reduced risk ({f['impact']}, feature {f['feature']})")
    if neighbors:
        reasons.append(f"Connected to {len(neighbors)} important neighboring transactions")

    summary_records.append({
        "txId":                  txid,
        "risk":                  round(float(row["prob"]), 4),
        "prediction":            "suspicious" if row["pred"]==1 else "normal",
        "important_neighbors":   neighbors[:10],
        "reasons":               reasons,
        "top_positive_factors":  pos_factors,
        "top_negative_factors":  neg_factors,
    })

with open(f"{VIZ_DIR}/explanation_summary.json","w") as f:
    json.dump(summary_records, f, indent=2)
print(f"explanation_summary.json: {len(summary_records)} records")


fraud_network.json: 203769 nodes, 9445 edges
explanation_summary.json: 11184 records


## 6. Website Export — no `.pt` files, pure JSON/CSV

In [13]:
# nodes.json — all nodes with risk metadata, frontend-ready
with open(f"{WEB_DIR}/nodes.json","w") as f:
    json.dump(viz_nodes, f)

# edges.json — test-set edges
with open(f"{WEB_DIR}/edges.json","w") as f:
    json.dump(viz_edges, f)

# predictions.json — all three model outputs per txId
pred_xgb_lu  = xgb_df.set_index("txId").rename(columns={"prob":"xgb_prob","pred":"xgb_pred"})
pred_gat_lu  = gatv2_df.set_index("txId").rename(columns={"prob":"gat_prob","pred":"gat_pred"})
pred_hyb_lu  = hybrid_df.set_index("txId").rename(columns={"prob":"hybrid_prob","pred":"hybrid_pred"})
pred_merged  = pred_xgb_lu[["xgb_prob","xgb_pred"]].join(
                   pred_gat_lu[["gat_prob","gat_pred"]], how="outer").join(
                   pred_hyb_lu[["hybrid_prob","hybrid_pred","true_label"]], how="outer").reset_index()
with open(f"{WEB_DIR}/predictions.json","w") as f:
    json.dump(pred_merged.to_dict(orient="records"), f)

# graph_summary.json — quick stats for a dashboard header
n_susp = int((hybrid_df["pred"]==1).sum())
n_norm = int((hybrid_df["pred"]==0).sum())
graph_summary = {
    "total_test_transactions": len(hybrid_df),
    "suspicious":  n_susp,
    "normal":      n_norm,
    "threshold":   THRESHOLD,
    "avg_risk_suspicious": round(float(hybrid_df[hybrid_df["pred"]==1]["prob"].mean()),4),
    "avg_risk_normal":     round(float(hybrid_df[hybrid_df["pred"]==0]["prob"].mean()),4),
    "explained_transactions": len(expl_df),
    "gnn_explained_targets":  len(target_info),
}
with open(f"{WEB_DIR}/graph_summary.json","w") as f:
    json.dump(graph_summary, f, indent=2)
print(json.dumps(graph_summary, indent=2))

# explanations/ — one JSON per explained transaction (for lazy loading in the frontend)
for rec in summary_records:
    if rec["reasons"]:   # only write files for transactions that have explanation data
        fname = f"{WEB_EXPL}/{rec['txId']}.json"
        with open(fname,"w") as f:
            json.dump(rec, f, indent=2)

print(f"\nWebsite files written to {WEB_DIR}")
print(f"  nodes.json, edges.json, predictions.json, graph_summary.json")
print(f"  explanations/  ({len(summary_records)} files)")


{
  "total_test_transactions": 11184,
  "suspicious": 437,
  "normal": 10747,
  "threshold": 0.8,
  "avg_risk_suspicious": 0.9567,
  "avg_risk_normal": 0.081,
  "explained_transactions": 25,
  "gnn_explained_targets": 20
}

Website files written to /content/drive/MyDrive/GraphShields/results_final/website
  nodes.json, edges.json, predictions.json, graph_summary.json
  explanations/  (11184 files)
